# Practice — 2026-11-04 — ML Testing & MLOps: Data Contracts, Reproducibility, and Performance-Regression Tests

**Dataset:** [California Housing](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) — 20,640 real 1990 California census block-group records (median income, house age, average rooms/bedrooms, population, occupancy, latitude/longitude) with median house value as the target. Loaded via `sklearn.datasets.fetch_california_housing`.

**Scenario:** You've joined a team as an MLOps/QA engineer. They ship a median-house-value regression model through a nightly batch-scoring pipeline. Nobody currently checks (a) whether the data flowing into the pipeline still matches what the model expects, (b) whether retraining is actually deterministic, or (c) whether a newly retrained model is any good before it replaces the one in production. Your job is to write the automated checks that would run in CI before any retrained model is promoted.

**What you should produce** — three standalone, pure-Python test functions plus a tiny runner that prints a CI-style pass/fail report:

1. `validate_schema(df, schema)` — a **data contract test**. Given a dict schema describing expected    columns, dtypes, and (for numeric columns) a valid `(min, max)` range, return a list of human-readable    violation strings (empty list = pass). It must catch: a missing column, an unexpected extra column, a    wrong dtype, and any value outside the declared range — without raising an exception on bad input.
2. `check_reproducibility(build_model_fn, X_train, y_train, X_test, y_test, n_runs=3, tol=1e-6)` — train the    model `n_runs` times with the exact same fixed `random_state`/hyperparameters and assert the test-set MAE    is identical (within `tol`) across runs. This is meant to catch accidental non-determinism (e.g. a step    that silently reshuffles rows or uses an unseeded random draw).
3. `check_performance_regression(candidate_mae, candidate_r2, baseline_mae, baseline_r2, mae_tol=0.05, r2_tol=0.02)`    — compare a newly trained ("candidate") model's metrics against a stored "golden" baseline from the    currently-deployed model, and return `(passed: bool, message: str)`. Fail if MAE is more than `mae_tol`    *relatively* worse than baseline, or if R² drops by more than `r2_tol` *absolute*.

Finally, write `run_ci_suite(...)` that calls all three checks against: (a) a deliberately corrupted copy of the test data, (b) a properly trained `RandomForestRegressor`, and (c) a deliberately **degraded** candidate model (trained on far too little signal) that the regression-performance check should catch and fail. Print a report that looks like a CI log, e.g.:
```
[PASS] schema contract
[PASS] reproducibility
[FAIL] performance regression: candidate MAE 0.91 vs baseline 0.53 (+71.7%, tol 5.0%)
```


## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42


## Load the data

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.rename(columns={"MedHouseVal": "target"})

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

df.head()


## Part 1 — Data contract test

Define the schema the pipeline expects, then implement `validate_schema`. A reasonable schema entry per column is `{"dtype": <numpy/pandas dtype kind>, "min": <float>, "max": <float>}`. You decide the exact representation — just be consistent with what you test below.


In [ ]:
# TODO: build a schema dict describing each column in X (dtype + plausible min/max),
# derived from the real training data's observed ranges.
SCHEMA = {
    # "MedInc": {"dtype": "float64", "min": 0.0, "max": 20.0},
    # ...
}


def validate_schema(frame, schema):
    """Return a list of violation strings; empty list means the frame passed.

    Must catch, without raising:
      - a column declared in `schema` that is missing from `frame`
      - a column present in `frame` but not declared in `schema`
      - a column whose dtype doesn't match the schema
      - any value outside the declared [min, max] range
    """
    violations = []
    # TODO: implement
    return violations


## Part 2 — Reproducibility test

Retrain the same model architecture with the same fixed `random_state` several times and confirm the test-set MAE doesn't drift between runs.


In [ ]:
def build_model():
    # TODO: return a freshly-constructed RandomForestRegressor with a FIXED random_state
    # and a small-ish number of estimators (keep it fast for a CI-style check).
    raise NotImplementedError


def check_reproducibility(build_model_fn, X_train, y_train, X_test, y_test, n_runs=3, tol=1e-6):
    """Train `n_runs` fresh models via build_model_fn and compare test-set MAE across runs.

    Return (passed: bool, message: str).
    """
    # TODO: implement
    raise NotImplementedError


## Part 3 — Performance regression test

Compare a candidate model's metrics against a stored "golden" baseline (as if read from a model registry) and flag a regression.


In [ ]:
def check_performance_regression(candidate_mae, candidate_r2, baseline_mae, baseline_r2,
                                   mae_tol=0.05, r2_tol=0.02):
    """Return (passed: bool, message: str).

    Fail if candidate_mae is more than `mae_tol` RELATIVELY worse than baseline_mae,
    or candidate_r2 is more than `r2_tol` ABSOLUTE worse than baseline_r2.
    """
    # TODO: implement
    raise NotImplementedError


## Run the CI suite

Wire everything together:
- Corrupt a copy of `X_test` (e.g. inject an out-of-range value and drop/rename a column) and confirm   `validate_schema` catches it.
- Train the "golden" baseline model (reasonable hyperparameters) and record its MAE/R² on `X_test`/`y_test`.
- Run `check_reproducibility` against the baseline's `build_model` function.
- Train a deliberately degraded "candidate" model (e.g. `max_depth=1`, or trained on only one or two   columns) and run `check_performance_regression` against the baseline metrics — it should **fail**.
- Print a small CI-style report for all three checks.


In [ ]:
def run_ci_suite():
    # TODO: implement, printing lines like:
    # [PASS] schema contract
    # [PASS] reproducibility
    # [FAIL] performance regression: candidate MAE 0.91 vs baseline 0.53 (+71.7%, tol 5.0%)
    pass


run_ci_suite()


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

**Approach.** Each check is a small, dependency-free function that returns data (a list of violations, or a
`(passed, message)` tuple) instead of raising or printing directly — that's what makes them composable into a
CI report and independently unit-testable.

```python
SCHEMA = {
    "MedInc":     {"dtype": "float64", "min": 0.0,    "max": 20.0},
    "HouseAge":   {"dtype": "float64", "min": 0.0,    "max": 60.0},
    "AveRooms":   {"dtype": "float64", "min": 0.0,    "max": 150.0},
    "AveBedrms":  {"dtype": "float64", "min": 0.0,    "max": 40.0},
    "Population": {"dtype": "float64", "min": 0.0,    "max": 40000.0},
    "AveOccup":   {"dtype": "float64", "min": 0.0,    "max": 1300.0},
    "Latitude":   {"dtype": "float64", "min": 32.0,   "max": 42.5},
    "Longitude":  {"dtype": "float64", "min": -125.0, "max": -113.0},
}


def validate_schema(frame, schema):
    violations = []

    missing = [c for c in schema if c not in frame.columns]
    for c in missing:
        violations.append(f"missing column: {c!r}")

    extra = [c for c in frame.columns if c not in schema]
    for c in extra:
        violations.append(f"unexpected column: {c!r}")

    for col, spec in schema.items():
        if col not in frame.columns:
            continue  # already reported above

        actual_dtype = str(frame[col].dtype)
        if actual_dtype != spec["dtype"]:
            violations.append(
                f"column {col!r} has dtype {actual_dtype}, expected {spec['dtype']}"
            )

        series = frame[col]
        lo, hi = spec["min"], spec["max"]
        bad = series[(series < lo) | (series > hi)]
        if len(bad) > 0:
            violations.append(
                f"column {col!r} has {len(bad)} value(s) outside [{lo}, {hi}] "
                f"(e.g. {bad.iloc[0]})"
            )

    return violations
```

```python
def build_model():
    return RandomForestRegressor(
        n_estimators=50, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1
    )


def check_reproducibility(build_model_fn, X_train, y_train, X_test, y_test, n_runs=3, tol=1e-6):
    maes = []
    for _ in range(n_runs):
        model = build_model_fn()
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        maes.append(mean_absolute_error(y_test, preds))

    spread = max(maes) - min(maes)
    passed = spread <= tol
    message = (
        f"MAE across {n_runs} runs: {[round(m, 6) for m in maes]}, "
        f"spread={spread:.2e} (tol={tol:.1e})"
    )
    return passed, message
```

```python
def check_performance_regression(candidate_mae, candidate_r2, baseline_mae, baseline_r2,
                                   mae_tol=0.05, r2_tol=0.02):
    mae_pct_increase = (candidate_mae - baseline_mae) / baseline_mae
    r2_drop = baseline_r2 - candidate_r2

    mae_ok = mae_pct_increase <= mae_tol
    r2_ok = r2_drop <= r2_tol
    passed = mae_ok and r2_ok

    message = (
        f"candidate MAE {candidate_mae:.3f} vs baseline {baseline_mae:.3f} "
        f"({mae_pct_increase:+.1%}, tol {mae_tol:.1%}); "
        f"candidate R2 {candidate_r2:.3f} vs baseline {baseline_r2:.3f} "
        f"(drop {r2_drop:+.3f}, tol {r2_tol:.3f})"
    )
    return passed, message
```

```python
def run_ci_suite():
    report = []

    # --- 1. Data contract test, run against a deliberately corrupted frame ---
    corrupted = X_test.copy()
    corrupted.loc[corrupted.index[0], "Latitude"] = 500.0   # out of range
    corrupted = corrupted.drop(columns=["Longitude"])        # missing column
    corrupted["extra_col"] = 1                               # unexpected column

    violations = validate_schema(corrupted, SCHEMA)
    schema_passed = len(violations) > 0  # we WANT it to catch the corruption
    report.append(("schema contract", schema_passed,
                    "; ".join(violations) if violations else "no violations found (unexpected!)"))

    # Sanity check: a clean frame should pass with zero violations.
    clean_violations = validate_schema(X_test, SCHEMA)
    assert clean_violations == [], f"clean data should pass schema check, got: {clean_violations}"

    # --- 2. Train the golden baseline model and record its metrics ---
    baseline_model = build_model()
    baseline_model.fit(X_train, y_train)
    baseline_preds = baseline_model.predict(X_test)
    baseline_mae = mean_absolute_error(y_test, baseline_preds)
    baseline_r2 = r2_score(y_test, baseline_preds)

    # --- 3. Reproducibility test on the baseline's build function ---
    repro_passed, repro_msg = check_reproducibility(build_model, X_train, y_train, X_test, y_test)
    report.append(("reproducibility", repro_passed, repro_msg))

    # --- 4. Train a deliberately degraded candidate and check for regression ---
    degraded_model = RandomForestRegressor(n_estimators=50, max_depth=1, random_state=RANDOM_STATE)
    degraded_model.fit(X_train[["MedInc"]], y_train)  # starve it of signal on purpose
    degraded_preds = degraded_model.predict(X_test[["MedInc"]])
    degraded_mae = mean_absolute_error(y_test, degraded_preds)
    degraded_r2 = r2_score(y_test, degraded_preds)

    perf_passed, perf_msg = check_performance_regression(
        degraded_mae, degraded_r2, baseline_mae, baseline_r2
    )
    report.append(("performance regression", perf_passed, perf_msg))

    for name, passed, msg in report:
        tag = "PASS" if passed else "FAIL"
        print(f"[{tag}] {name}: {msg}")


run_ci_suite()
```

**Why this design matters for ML quality work:**
- `validate_schema` returning a list (not raising) means you can log *every* violation in one pass instead
  of stopping at the first `KeyError`, which is what you want from a real data-contract check feeding a
  monitoring dashboard.
- The reproducibility check targets a specific, common bug class: forgetting to seed something (a shuffle, a
  subsample, a non-deterministic GPU op), which silently makes "retrain and redeploy" non-repeatable and
  breaks audits.
- The regression check is deliberately asymmetric (relative tolerance on error, absolute tolerance on R²)
  because a flat percentage tolerance on a metric that's already near zero (or already near its ceiling of 1
  for R²) is a common way these checks end up either too loose or always failing.

</details>
